In [61]:
with open('test.txt', 'r') as f:
    data = f.read().splitlines()

print('\n'.join(data))

#.#####################
#.......#########...###
#######.#########.#.###
###.....#.>.>.###.#.###
###v#####.#v#.###.#.###
###.>...#.#.#.....#...#
###v###.#.#.#########.#
###...#.#.#.......#...#
#####.#.#.#######.#.###
#.....#.#.#.......#...#
#.#####.#.#.#########v#
#.#...#...#...###...>.#
#.#.#v#######v###.###v#
#...#.>.#...>.>.#.###.#
#####v#.#.###v#.#.###.#
#.....#...#...#.#.#...#
#.#########.###.#.#.###
#...###...#...#...#.###
###.###.#.###v#####v###
#...#...#.#.>.>.#.>.###
#.###.###.#.###.#.#v###
#.....###...###...#...#
#####################.#


In [62]:
TREE = '#'
PATH = '.'
SLOPES = {
    '^': (0, -1),
    '>': (1, 0),
    'v': (0, 1),
    '<': (-1, 0)
}

DIRECTIONS = list(SLOPES.values())

START = (1, 0)

In [63]:
def find_paths(data, start, icy=True):
    paths = []
    tasks = [(start, (0, 1), [start])]

    while tasks:
        pos, direction, current_path = tasks.pop()
        x, y = pos
        direction_x, direction_y = direction
        next_x, next_y = x + direction_x, y + direction_y

        if (
            next_x < 0 or 
            next_x >= len(data[0]) or 
            next_y < 0 or 
            next_y >= len(data)
        ):
            continue

        if (next_x, next_y) in current_path:
            continue

        next_step = data[next_y][next_x]

        if next_step == TREE:
            continue

        new_path = current_path + [(next_x, next_y)]

        if next_y == len(data) - 1:
            paths.append(new_path)
            continue

        if icy:
            if next_step == PATH:
                for d in DIRECTIONS:
                    tasks.append(((next_x, next_y), d, new_path))
            elif next_step in SLOPES and SLOPES[next_step] == direction:
                tasks.append(((next_x, next_y), SLOPES[next_step], new_path))
        else:
            if next_step == PATH or next_step in SLOPES:
                for d in DIRECTIONS:
                    tasks.append(((next_x, next_y), d, new_path))

    return paths

paths = find_paths(data, START)
print(max(len(p) - 1 for p in paths))

94


In [64]:
with open('input.txt', 'r') as f:
    data = f.read().splitlines()

In [65]:
paths_1 = find_paths(data=data, start=START, icy=True)
max_path_1 = max(len(p) - 1 for p in paths_1)

print(f"Part 1: {max_path_1}")

Part 1: 2034


In [70]:
def open_neighbors(pos):
    x, y = pos

    return [
        (x + dx, y + dy)
        for dx, dy in DIRECTIONS
        if 0 <= y + dy < len(data)
        and 0 <= x + dx < len(data[0])
        and data[y + dy][x + dx] != TREE
    ]


def depth_first_search(start):
    end = next((x, len(data) - 1) for x, c in enumerate(data[-1]) if c != TREE)

    junctions = {start, end}

    for y, row in enumerate[str](data):
        for x, cell in enumerate[str](row):
            if cell == TREE:
                continue
            
            neighbors = open_neighbors((x, y))
            
            if len(neighbors) >= 3:
                junctions.add((x, y))

    print("Found", len(junctions), "junctions")

    graph = {j: [] for j in junctions}

    for junction in junctions:
        for first_step in open_neighbors(junction):
            previous_step = junction
            current_step = first_step
            step_count = 1

            while current_step not in junctions:
                neighbors = open_neighbors(current_step)
                next_step = next(n for n in neighbors if n != previous_step)

                previous_step = current_step
                current_step = next_step
                step_count += 1
                
            graph[junction].append((current_step, step_count))

    best = 0
    tasks = [(start, {start}, 0)]

    path_count = 0

    while tasks:
        node, visited, length = tasks.pop()

        if node == end:
            best = max(best, length)
            path_count += 1
            continue

        for neighbor, step_count in graph[node]:
            if neighbor in visited:
                continue

            tasks.append((neighbor, visited.union({neighbor}), length + step_count))

    print("Found", path_count, "paths")
    return best


print(f"Part 2: {depth_first_search(START)}")

Found 36 junctions
Found 1262816 paths
Part 2: 6302
